In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# --- 1. Load Data
df_sales = pd.read_csv("sales.csv")

In [3]:

df_sales["date"] = pd.to_datetime(df_sales["date"])
df_sales = df_sales.sort_values("date").reset_index(drop=True)

# Feature Engineering
df_sales["year"] = df_sales["date"].dt.year
df_sales["month"] = df_sales["date"].dt.month
df_sales["day"] = df_sales["date"].dt.day
df_sales["is_state_holiday"] = (
    (df_sales["state_holiday"].astype(str).str.strip() != "0")
).astype(int)

In [4]:
# Train only on open days with actual sales
df_trainable = df_sales[(df_sales["open"] == 1) & (df_sales["sales"] > 0)].copy()

X = df_trainable.drop(columns=["sales", "date", "state_holiday", "open"])
y = df_trainable["sales"]

In [5]:
# --- 2. Train / Validation / Test Sequential Split (70% / 15% / 15%) ---
total_rows = len(df_trainable)
#calculates the row index where the first 70% of your data ends.
train_cutoff = int(total_rows * 0.70)

#calculates the row index where 85% of your total data ends.
val_cutoff = int(total_rows * 0.85)

X_train = X.iloc[:train_cutoff].copy()
X_val = X.iloc[train_cutoff:val_cutoff].copy()
X_test = X.iloc[val_cutoff:].copy()

y_train = y.iloc[:train_cutoff]
y_val = y.iloc[train_cutoff:val_cutoff]
y_test = y.iloc[val_cutoff:]

In [6]:
# --- 3. Target Encoding on store_ID (Using y_train only!) ---
store_means = y_train.groupby(X_train["store_ID"]).mean()
global_mean = y_train.mean()

# Map to all three splits
for df in [X_train, X_val, X_test]:
    df["store_avg_sales"] = df["store_ID"].map(store_means).fillna(global_mean)
    df.drop(columns=["store_ID"], inplace=True)

In [7]:
# --- 4. Scale Features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [8]:
# --- 5. Define Models ---
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=15, random_state=42, n_jobs=-1
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42
    ),
}

In [9]:
# --- 6. Train on 'Train' and Evaluate on 'Validation' ---
print("=== VALIDATION PHASE (Tuning Dashboard) ===")
for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    # Predict on Validation data
    y_val_pred = model.predict(X_val_scaled)

    # Metrics
    r2_v = r2_score(y_val, y_val_pred)
    mae_v = mean_absolute_error(y_val, y_val_pred)
    rmse_v = np.sqrt(mean_squared_error(y_val, y_val_pred))

    print(f"\n{name}:")
    print(f"  R² Score:  {r2_v:.4f}")
    print(f"  MAE:       ${mae_v:,.2f}")
    print(f"  RMSE:      ${rmse_v:,.2f}")

=== VALIDATION PHASE (Tuning Dashboard) ===

Linear Regression:
  R² Score:  0.7934
  MAE:       $1,019.23
  RMSE:      $1,454.02

Random Forest:
  R² Score:  0.9054
  MAE:       $687.58
  RMSE:      $983.78

XGBoost:
  R² Score:  0.9002
  MAE:       $723.34
  RMSE:      $1,010.42


In [ ]:
# --- 7. Final Test Phase (Only on your Champion Model) ---
# Assuming Random Forest wins validation, run final verification on Test:
champion_name = "Random Forest"
champion_model = models[champion_name]
y_test_pred = champion_model.predict(X_test_scaled)

print(f"\n=== FINAL TEST PHASE (Unseen Future Data) ===")
print(f"Champion Model: {champion_name}")
print(f"  Final R² Score: {r2_score(y_test, y_test_pred):.4f}")
print(f"  Final MAE:      ${mean_absolute_error(y_test, y_test_pred):,.2f}")


=== FINAL TEST PHASE (Unseen Future Data) ===
Champion Model: Random Forest
  Final R² Score: 0.8915
  Final MAE:      $713.27
